### 2.1 理论计算题解答

给定字符序列："ababc"
词汇表：V = {'a', 'b', 'c'}，|V| = 3

#### 步骤1：统计字符转移次数

从序列 "ababc" 中提取相邻字符对：
- a→b (第1-2位)
- b→a (第2-3位)
- a→b (第3-4位)
- b→c (第4-5位)

统计结果：
- count(a→b) = 2
- count(b→a) = 1
- count(b→c) = 1
- 其他转移次数均为0

#### 步骤2：统计每个字符作为前一个字符的总次数

- count(a) = count(a→a) + count(a→b) + count(a→c) = 0 + 2 + 0 = 2
- count(b) = count(b→a) + count(b→b) + count(b→c) = 1 + 0 + 1 = 2
- count(c) = count(c→a) + count(c→b) + count(c→c) = 0 + 0 + 0 = 0

#### 步骤3：应用拉普拉斯平滑公式

拉普拉斯平滑（加1平滑）公式：
$$ p(w_t | w_{t-1}) = \frac{count(w_{t-1} \rightarrow w_t) + 1}{count(w_{t-1}) + |V|} $$

1. **计算 p('a' | 'b')**：
$$ p('a' | 'b') = \frac{count(b \rightarrow a) + 1}{count(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4 $$

2. **计算 p('c' | 'b')**：
$$ p('c' | 'b') = \frac{count(b \rightarrow c) + 1}{count(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4 $$

#### 最终答案
1. $p('a' | 'b') = 0.4$
2. $p('c' | 'b') = 0.4$

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数，用于自回归语言模型的数据准备
    
    参数:
        text (str): 输入文本
        n (int): 滑动窗口大小
    
    返回:
        vocab (dict): 词汇表 {词: ID}
        features (list): 特征序列列表
        labels (list): 标签列表
    """
    # 1. 将文本转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(words) - n + 1):
        feature = words[i:i+n]
        features.append(feature)
        
        if i + n < len(words):
            label = words[i+n]
            labels.append(label)
        else:
            labels.append(None)
    
    return vocab, features, labels

# 测试示例
if __name__ == "__main__":
    test_text = "The time machine"
    vocab, features, labels = preprocess_text(test_text, n=2)
    print("词汇表:", vocab)
    print("特征列表:", features)
    print("标签列表:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time'], ['time', 'machine']]
标签列表: ['machine', None]


### 3.1 理论计算题解答

#### 问题描述
考虑线性RNN（无偏置）：
$$ h_t = W_{hh}h_{t-1} + W_{hx}x_t $$
$$ o_t = W_{oh}h_t $$
损失函数：
$$ L = \frac{1}{2}\sum_{t=1}^T (o_t - y_t)^2 $$

#### 步骤1：反向传播计算梯度

首先定义 $\delta_t = \frac{\partial L}{\partial h_t}$，则：

对于最后一个时间步 $t=T$：
$$ \delta_T = \frac{\partial L}{\partial h_T} = W_{oh}^\top (o_T - y_T) $$

对于中间时间步 $t<T$：
$$ \delta_t = \frac{\partial L}{\partial h_t} = W_{hh}^\top \delta_{t+1} + W_{oh}^\top (o_t - y_t) $$

#### 步骤2：计算 $\frac{\partial L}{\partial W_{hh}}$

利用链式法则，将所有时间步的梯度相加：
$$ \frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \frac{\partial L}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}} = \sum_{t=1}^T \delta_t h_{t-1}^\top $$

其中 $h_0$ 为初始隐藏状态。

#### 步骤3：梯度消失或爆炸的条件

展开 $\delta_t$ 的递推公式：
$$ \delta_t = \sum_{k=t}^T (W_{hh}^\top)^{k-t} W_{oh}^\top (o_k - y_k) $$

当 $W_{hh}$ 的谱半径（最大特征值）$\rho(W_{hh}) < 1$ 时，$(W_{hh}^\top)^{k-t}$ 随 $k-t$ 增大呈指数衰减 → **梯度消失**

当 $\rho(W_{hh}) > 1$ 时，$(W_{hh}^\top)^{k-t}$ 随 $k-t$ 增大呈指数增长 → **梯度爆炸**

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前输入 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态 (batch_size, hidden_size)
        W_hh: 隐藏状态权重 (hidden_size, hidden_size)
        W_hx: 输入权重 (hidden_size, input_size)
        b_h: 偏置 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态
        cache: 缓存中间变量用于反向传播
    """
    h_t = np.tanh(np.dot(h_prev, W_hh.T) + np.dot(x_t, W_hx.T) + b_h)
    cache = (x_t, h_prev, W_hh, W_hx, b_h, h_t)
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播（仅计算梯度）
    
    参数:
        dh_next: 对当前隐藏状态h_t的梯度 (batch_size, hidden_size)
        cache: 前向传播的缓存
    
    返回:
        dx_t: 对输入x_t的梯度
        dh_prev: 对上一隐藏状态h_prev的梯度
        dW_hx: 对W_hx的梯度
        dW_hh: 对W_hh的梯度
        db_h: 对b_h的梯度
    """
    x_t, h_prev, W_hh, W_hx, b_h, h_t = cache
    
    # tanh的导数: 1 - tanh²
    dtanh = dh_next * (1 - h_t ** 2)
    
    # 计算梯度
    dx_t = np.dot(dtanh, W_hx)
    dh_prev = np.dot(dtanh, W_hh)
    dW_hx = np.dot(dtanh.T, x_t)
    dW_hh = np.dot(dtanh.T, h_prev)
    db_h = np.sum(dtanh, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试示例
if __name__ == "__main__":
    # 设置参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 初始化随机数据
    np.random.seed(42)
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    W_hx = np.random.randn(hidden_size, input_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_hx, b_h)
    print("前向传播结果 h_t:")
    print(h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    print("\n反向传播梯度:")
    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("db_h shape:", db_h.shape)

前向传播结果 h_t:
[[-0.99660809 -0.99991936 -0.98976703 -0.96533118]
 [-0.84763773  0.04633205 -0.10191286  0.13724861]]

反向传播梯度:
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (4, 3)
dW_hh shape: (4, 4)
db_h shape: (4,)


### 4.1 理论计算题解答

#### 问题描述
深度双向RNN，L层，每层隐藏单元数为H，输入维度为D，输出维度为O。

#### 参数计算

对于双向RNN，每层包含前向和后向两个方向。

**第1层（输入层）的参数：**
- 前向RNN：
  - 输入权重：$W_{hx}^{(1,\text{forward})}$ 维度 $H \times D$
  - 隐藏状态权重：$W_{hh}^{(1,\text{forward})}$ 维度 $H \times H$
  - 偏置：$b_h^{(1,\text{forward})}$ 维度 $H$
- 后向RNN：
  - 输入权重：$W_{hx}^{(1,\text{backward})}$ 维度 $H \times D$
  - 隐藏状态权重：$W_{hh}^{(1,\text{backward})}$ 维度 $H \times H$
  - 偏置：$b_h^{(1,\text{backward})}$ 维度 $H$

**第l层（2 ≤ l ≤ L）的参数：**
- 前向RNN：
  - 输入权重：$W_{hx}^{(l,\text{forward})}$ 维度 $H \times 2H$
  - 隐藏状态权重：$W_{hh}^{(l,\text{forward})}$ 维度 $H \times H$
  - 偏置：$b_h^{(l,\text{forward})}$ 维度 $H$
- 后向RNN：
  - 输入权重：$W_{hx}^{(l,\text{backward})}$ 维度 $H \times 2H$
  - 隐藏状态权重：$W_{hh}^{(l,\text{backward})}$ 维度 $H \times H$
  - 偏置：$b_h^{(l,\text{backward})}$ 维度 $H$

**输出层参数：**
- 输出权重：$W_{oh}$ 维度 $O \times 2H$
- 输出偏置：$b_o$ 维度 $O$

#### 总参数数量

第1层总参数：
$$ 2 \times (H \times D + H \times H + H) = 2H(D + H + 1) $$

第2层到第L层总参数：
$$ (L-1) \times 2 \times (H \times 2H + H \times H + H) = 2(L-1)H(3H + 1) $$

输出层总参数：
$$ O \times 2H + O = O(2H + 1) $$

**总参数表达式：**
$$ \boxed{2H(D + H + 1) + 2(L-1)H(3H + 1) + O(2H + 1)} $$

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        """
        双向RNN编码器
        
        参数:
            input_dim: 输入维度
            hidden_dim: 隐藏层维度（每个方向）
            num_layers: RNN层数
        """
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False
        )
    
    def forward(self, x):
        """
        前向传播
        
        参数:
            x: 输入序列 (seq_len, batch, input_dim)
        
        返回:
            all_hidden: 每个时间步拼接的前向和后向隐藏状态 (seq_len, batch, 2*hidden_dim)
            final_hidden: 最终时间步拼接的隐藏状态 (作为序列列表表示)
        """
        # RNN前向传播
        all_hidden, _ = self.rnn(x)
        
        # 获取最终时间步的前向和后向隐藏状态
        seq_len = x.size(0)
        batch_size = x.size(1)
        
        # 前向最后一步: all_hidden[-1, :, :hidden_dim]
        # 后向最后一步: all_hidden[0, :, hidden_dim:]
        forward_final = all_hidden[-1, :, :self.hidden_dim]
        backward_final = all_hidden[0, :, self.hidden_dim:]
        final_hidden = torch.cat([forward_final, backward_final], dim=1)
        
        # 转换为序列列表表示（每个样本一个张量）
        final_hidden_list = [final_hidden[i] for i in range(batch_size)]
        
        return all_hidden, final_hidden_list

# 测试示例
if __name__ == "__main__":
    # 设置参数
    seq_len = 5
    batch_size = 2
    input_dim = 3
    hidden_dim = 4
    
    # 初始化模型和随机输入
    model = BidirectionalRNNEncoder(input_dim, hidden_dim)
    x = torch.randn(seq_len, batch_size, input_dim)
    
    # 前向传播
    all_hidden, final_hidden_list = model(x)
    
    print("输入形状:", x.shape)
    print("所有时间步隐藏状态形状:", all_hidden.shape)
    print("最终隐藏状态列表长度:", len(final_hidden_list))
    print("每个最终隐藏状态形状:", final_hidden_list[0].shape)
    print("\n第一个样本的最终隐藏状态:")
    print(final_hidden_list[0])

输入形状: torch.Size([5, 2, 3])
所有时间步隐藏状态形状: torch.Size([5, 2, 8])
最终隐藏状态列表长度: 2
每个最终隐藏状态形状: torch.Size([8])

第一个样本的最终隐藏状态:
tensor([ 0.5236, -0.3708,  0.4370,  0.8697,  0.6748,  0.5303, -0.2149, -0.8673],
       grad_fn=<SelectBackward0>)


### 5.1 理论计算题解答

#### Skip-gram 负采样损失函数推导

**问题设定：**
- 中心词 $w_c$，上下文词 $w_o$
- 词向量 $\mathbf{v}_c$（中心词向量），$\mathbf{u}_o$（上下文词向量）
- 采样 $K$ 个负样本 $\{w_{k}\}_{k=1}^K$

**步骤1：单个正样本的概率**
对于中心词 $w_c$ 和上下文词 $w_o$，它们共同出现的概率为：
$$ P(D=1 | w_c, w_o) = \sigma(\mathbf{u}_o^\top \mathbf{v}_c) = \frac{1}{1 + e^{-\mathbf{u}_o^\top \mathbf{v}_c}} $$

**步骤2：单个负样本的概率**
对于负样本 $w_k$，它们不共同出现的概率为：
$$ P(D=0 | w_c, w_k) = 1 - \sigma(\mathbf{u}_k^\top \mathbf{v}_c) = \sigma(-\mathbf{u}_k^\top \mathbf{v}_c) $$

**步骤3：对数似然损失**
单个样本的对数似然为：
$$ \log P(D=1 | w_c, w_o) + \sum_{k=1}^K \log P(D=0 | w_c, w_k) $$

**步骤4：损失函数（对数似然的负值）**
$$ L = -\left[ \log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) + \sum_{k=1}^K \log \sigma(-\mathbf{u}_k^\top \mathbf{v}_c) \right] $$

#### 负采样方法

负样本从噪声分布 $P_n(w)$ 中采样，通常使用词频的 $3/4$ 次幂归一化：
$$ P_n(w) = \frac{count(w)^{3/4}}{\sum_{w' \in V} count(w')^{3/4}} $$

可以通过以下方式采样：
1. 构建一个大的数组（长度为10^8），按 $P_n(w)$ 比例填充各词的索引
2. 每次随机采样数组中的一个位置即可获得负样本

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        """
        CBOW模型
        
        参数:
            vocab_size: 词汇表大小
            embed_dim: 嵌入维度
        """
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        
        # 输入权重矩阵 W (V, d)
        self.W = nn.Embedding(vocab_size, embed_dim)
        # 输出权重矩阵 W_out (d, V)
        self.W_out = nn.Linear(embed_dim, vocab_size, bias=False)
    
    def forward(self, context_indices):
        """
        前向传播
        
        参数:
            context_indices: 上下文词索引列表 (batch, context_size)
        
        返回:
            loss: 交叉熵损失
            log_probs: 对数概率 (batch, vocab_size)
        """
        batch_size = context_indices.size(0)
        
        # 步骤1: 查找上下文词向量
        # context_embeds shape: (batch, context_size, embed_dim)
        context_embeds = self.W(context_indices)
        
        # 步骤2: 平均上下文向量作为隐藏层
        # h shape: (batch, embed_dim)
        h = torch.mean(context_embeds, dim=1)
        
        # 步骤3: 计算输出得分
        # scores shape: (batch, vocab_size)
        scores = self.W_out(h)
        
        # 步骤4: 计算对数概率
        log_probs = F.log_softmax(scores, dim=1)
        
        return log_probs
    
    def compute_loss(self, context_indices, target_indices):
        """
        计算交叉熵损失
        
        参数:
            context_indices: 上下文词索引 (batch, context_size)
            target_indices: 目标中心词索引 (batch,)
        
        返回:
            loss: 平均损失值
        """
        log_probs = self.forward(context_indices)
        # 负对数似然损失
        loss = F.nll_loss(log_probs, target_indices)
        return loss

# 测试示例
if __name__ == "__main__":
    # 设置参数
    vocab_size = 1000
    embed_dim = 50
    batch_size = 32
    context_size = 4  # 上下文词数量
    
    # 初始化模型
    model = CBOWModel(vocab_size, embed_dim)
    
    # 创建随机数据
    context_indices = torch.randint(0, vocab_size, (batch_size, context_size))
    target_indices = torch.randint(0, vocab_size, (batch_size,))
    
    # 计算损失
    loss = model.compute_loss(context_indices, target_indices)
    
    print("上下文索引形状:", context_indices.shape)
    print("目标索引形状:", target_indices.shape)
    print("CBOW损失:", loss.item())
    
    # 验证输出
    log_probs = model(context_indices)
    print("对数概率形状:", log_probs.shape)
    print("对数概率示例:", log_probs[0, :5])

上下文索引形状: torch.Size([32, 4])
目标索引形状: torch.Size([32])
CBOW损失: 6.975991249084473
对数概率形状: torch.Size([32, 1000])
对数概率示例: tensor([-6.6919, -7.0221, -7.0152, -7.1242, -7.2017], grad_fn=<SliceBackward0>)


### 6.1 理论计算题解答

#### 缩放点积注意力计算

**设定数值：**
为了演示，我们定义以下矩阵：

$$ Q = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{pmatrix} \in \mathbb{R}^{3 \times 4} $$

$$ K = \begin{pmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \\ 1 & 1 & 1 & 1 \end{pmatrix} \in \mathbb{R}^{3 \times 4} $$

$$ V = \begin{pmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{pmatrix} \in \mathbb{R}^{3 \times 5} $$

已知 $d_k = 4$，$\sqrt{d_k} = 2$。

**步骤1：计算得分矩阵 $QK^\top$**

$$ QK^\top = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \end{pmatrix} \begin{pmatrix} 1 & 0 & 1 \\ 0 & 1 & 1 \\ 1 & 0 & 1 \\ 0 & 1 & 1 \end{pmatrix} = \begin{pmatrix} 4 & 6 & 10 \\ 12 & 14 & 26 \\ 20 & 22 & 42 \end{pmatrix} $$

**步骤2：缩放 $\frac{QK^\top}{\sqrt{d_k}}$**

$$ \text{score} = \frac{QK^\top}{\sqrt{4}} = \begin{pmatrix} 2 & 3 & 5 \\ 6 & 7 & 13 \\ 10 & 11 & 21 \end{pmatrix} $$

**步骤3：Softmax归一化（按行）**

对于第一行 $[2, 3, 5]$：
$$ \text{softmax}([2, 3, 5]) = \left[ \frac{e^2}{e^2+e^3+e^5}, \frac{e^3}{e^2+e^3+e^5}, \frac{e^5}{e^2+e^3+e^5} \right] \approx [0.042, 0.114, 0.844] $$

同理可得完整的注意力权重矩阵：
$$ \text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V \approx \begin{pmatrix} 10.0 & 11.0 & 12.0 & 13.0 & 14.0 \\ 10.8 & 11.8 & 12.8 & 13.8 & 14.8 \\ 11.0 & 12.0 & 13.0 & 14.0 & 15.0 \end{pmatrix} $$

（注：最终输出维度为 $3 \times 5$）

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        """
        多头注意力机制
        
        参数:
            d_model: 模型维度
            num_heads: 注意力头数
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        缩放点积注意力
        """
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights
    
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数:
            X: 输入张量 (seq_len, batch, d_model)
            mask: 可选的掩码
        
        返回:
            output: 多头注意力输出 (seq_len, batch, d_model)
            attn_weights: 注意力权重
        """
        seq_len, batch_size, _ = X.size()
        
        # 步骤1: 线性投影得到Q, K, V
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 步骤2: 分割为多个头
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(0, 2).transpose(1, 2)
        
        # 步骤3: 计算缩放点积注意力
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # 步骤4: 拼接多头输出
        attn_output = attn_output.transpose(1, 2).transpose(0, 2).contiguous().view(seq_len, batch_size, self.d_model)
        
        # 步骤5: 最终线性投影
        output = self.W_o(attn_output)
        
        return output, attn_weights

# 测试示例
if __name__ == "__main__":
    # 设置参数
    seq_len = 5
    batch_size = 2
    d_model = 4
    num_heads = 2
    
    # 初始化模型和随机输入
    mha = MultiHeadAttention(d_model, num_heads)
    X = torch.randn(seq_len, batch_size, d_model)
    
    # 前向传播
    output, attn_weights = mha(X)
    
    print("输入形状:", X.shape)
    print("输出形状:", output.shape)
    print("注意力权重形状:", attn_weights.shape)
    print("\n输出张量示例:")
    print(output[0, 0, :])

输入形状: torch.Size([5, 2, 4])
输出形状: torch.Size([5, 2, 4])
注意力权重形状: torch.Size([2, 5, 2, 2])

输出张量示例:
tensor([0.3793, 0.5579, 0.0837, 0.7687], grad_fn=<SliceBackward0>)
